## 1. Định nghĩa thuật toán

Thuật toán gradient chính sách cơ bản nhất là thuật toán REINFORCE, còn gọi là thuật toán Monte-Carlo Policy Gradient. Mục tiêu tối ưu chính sách của ta như sau:

$$
J_{\theta}= \Psi_{\pi} \nabla_\theta \log \pi_\theta\left(a_t \mid s_t\right)
$$

Trong đó $\Psi_{\pi}$ trong thuật toán REINFORCE biểu thị lợi tức chiết khấu (công thức cụ thể xem mã giả), cũng có thể dùng lợi thế để ước lượng, tức thuật toán A3C mà ta quen thuộc, điều này sẽ được nói đến ở các phần sau kể cả thuật toán GAE.

### 1.1. Thiết kế hàm chính sách

Đã là gradient chính sách thì tính gradient trực tiếp trên hàm chính sách, vậy hàm chính sách thiết kế thế nào? Nói chung có hai cách thiết kế, một là hàm softmax, cái còn lại là phân phối Gauss $\mathbb{N}\left(\phi(\mathbb{s})^{\mathbb{\pi}} \theta, \sigma^2\right)$, cái trước dùng cho không gian hành động rời rạc, cái sau chủ yếu dùng cho không gian hành động liên tục.

Hàm softmax có thể biểu diễn là:
$$
\pi_\theta(s, a)=\frac{e^{\phi(s, a)^{T_\theta}}}{\sum_b e^{\phi(s, b)^{T^T}}}
$$
Gradient tương ứng là:
$$
\nabla_\theta \log \pi_\theta(s, a)=\phi(s, a)-\mathbb{E}_{\pi_\theta}[\phi(s,)
$$
Gradient tương ứng của phân phối Gauss là:
$$
\nabla_\theta \log \pi_\theta(s, a)=\frac{\left(a-\phi(s)^T \theta\right) \phi(s)}{\sigma^2}
$$
Nhưng với một số trường hợp đặc biệt, ví dụ trong ví dụ minh họa lần này số chiều hành động = 2 và là không gian rời rạc, lúc này có thể dùng phân phối Bernoulli để cài đặt, cách này thực ra không được khuyến khích, ở đây minh họa cho mọi người cũng là để thể hiện một số trường hợp đặc biệt, gợi mở cho mọi người một số suy nghĩ, ví dụ quan hệ giữa các phân phối Bernoulli, Binomial, Gaussian. Nói đơn giản, phân phối Binomial khi $n = 1$ chính là phân phối Bernoulli, khi $n \rightarrow \infty$ chính là phân phối Gaussian.




### 1.2. Thiết kế mô hình

Như đã nói ở trên, tuy ví dụ minh họa lần này là không gian rời rạc, nhưng vì số chiều hành động bằng 2, lúc này có thể dùng một phân phối đặc biệt để biểu diễn hàm chính sách, tức phân phối Bernoulli. Phân phối Bernoulli thực chất lấy một xác suất làm đầu vào, rồi lấy mẫu hành động từ đó, hành động lấy mẫu từ Bernoulli chỉ có thể là 0 hoặc 1, giống như mặt sấp mặt ngửa khi tung đồng xu. Trong trường hợp này, mô hình chính sách của ta cần trên cơ sở MLP, lấy trạng thái làm đầu vào, lấy hành động làm đầu ra của tầng áp chót, và thêm hàm kích hoạt ở tầng cuối để đưa ra xác suất của hành động tương ứng. Bạn nào chưa rõ tác dụng của hàm kích hoạt có thể xem lại kiến thức liên quan đến học sâu, nói đơn giản tác dụng của nó là tăng tính phi tuyến của mạng nơ-ron. Đã cần đưa ra xác suất của hành động tương ứng, thì giá trị đầu ra cần nằm trong khoảng 0-1, lúc này hàm sigmoid vừa hay đáp ứng nhu cầu của ta, code cài đặt tham khảo như sau.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class PGNet(nn.Module):
    def __init__(self, input_dim,output_dim,hidden_dim=128):
        """ Khởi tạo mạng q, là mạng kết nối đầy đủ
            input_dim: số đặc trưng đầu vào, tức số chiều trạng thái của môi trường
            output_dim: số chiều hành động đầu ra
        """
        super(PGNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim) # tầng đầu vào
        self.fc2 = nn.Linear(hidden_dim,hidden_dim) # tầng ẩn
        self.fc3 = nn.Linear(hidden_dim, output_dim) # tầng đầu ra
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        return x

### 1.3. Thiết kế hàm cập nhật

Như đã nhắc ở trên, mục tiêu tối ưu của ta, tức hàm loss của thuật toán gradient chính sách như sau:
$$
J_{\theta}= \Psi_{\pi} \nabla_\theta \log \pi_\theta\left(a_t \mid s_t\right)
$$

Ta cần tách thành hai phần $\Psi_{\pi}$ và $\nabla_\theta \log \pi_\theta\left(a_t \mid s_t\right)$ để tính riêng, trước hết xem phần hàm giá trị $\Psi_{\pi}$, trong thuật toán REINFORCE hàm giá trị là lợi tức chiết khấu bắt đầu từ thời điểm hiện tại, như sau:
$$
G \leftarrow \sum_{k=t+1}^{T} \gamma^{k-1} r_{k}
$$

Khi cài đặt bằng code thực tế điều này có thể hơi rối, ta có thể nhìn ngược lại, trong cùng một tập, thời điểm kết thúc của ta là $T$, thì lợi tức tương ứng $G_T=\gamma^{T-1}r_T$, còn $G_{T-1}=\gamma^{T-2}r_{T-1}+\gamma^{T-1}r_T$ tương ứng, ở đây trong code ta dùng một mẹo quy hoạch động, như sau:
```python
running_add = running_add * self.gamma + reward_pool[i] # giá trị ban đầu của running_add là 0
```
Công thức này cũng lặp ngược lại, giá trị lần thứ nhất bằng:
$$
running\_add = r_T
$$
Giá trị lần thứ hai bằng:
$$
running\_add = r_T*\gamma+r_{T-1}
$$
Giá trị lần thứ ba bằng:
$$
running\_add = (r_T*\gamma+r_{T-1})*\gamma+r_{T-2} = r_T*\gamma^2+r_{T-1}*\gamma+r_{T-2}
$$


In [ ]:
import torch
from torch.distributions import Bernoulli
from torch.autograd import Variable
import numpy as np

class PolicyGradient:
    
    def __init__(self, model,memory,cfg):
        self.gamma = cfg['gamma']
        self.device = torch.device(cfg['device']) 
        self.memory = memory
        self.policy_net = model.to(self.device)
        self.optimizer = torch.optim.RMSprop(self.policy_net.parameters(), lr=cfg['lr'])

    def sample_action(self,state):

        state = torch.from_numpy(state).float()
        state = Variable(state)
        probs = self.policy_net(state)
        m = Bernoulli(probs) # phân phối Bernoulli
        action = m.sample()
        
        action = action.data.numpy().astype(int)[0] # chuyển thành số vô hướng
        return action
    def predict_action(self,state):

        state = torch.from_numpy(state).float()
        state = Variable(state)
        probs = self.policy_net(state)
        m = Bernoulli(probs) # phân phối Bernoulli
        action = m.sample()
        action = action.data.numpy().astype(int)[0] # chuyển thành số vô hướng
        return action
        
    def update(self):
        state_pool,action_pool,reward_pool= self.memory.sample()
        state_pool,action_pool,reward_pool = list(state_pool),list(action_pool),list(reward_pool)
        # Discount reward
        running_add = 0
        for i in reversed(range(len(reward_pool))):
            if reward_pool[i] == 0:
                running_add = 0
            else:
                running_add = running_add * self.gamma + reward_pool[i]
                reward_pool[i] = running_add

        # Normalize reward
        reward_mean = np.mean(reward_pool)
        reward_std = np.std(reward_pool)
        for i in range(len(reward_pool)):
            reward_pool[i] = (reward_pool[i] - reward_mean) / reward_std

        # Gradient Desent
        self.optimizer.zero_grad()

        for i in range(len(reward_pool)):
            state = state_pool[i]
            action = Variable(torch.FloatTensor([action_pool[i]]))
            reward = reward_pool[i]
            state = Variable(torch.from_numpy(state).float())
            probs = self.policy_net(state)
            m = Bernoulli(probs)
            loss = -m.log_prob(action) * reward  # Negtive score function x reward
            # print(loss)
            loss.backward()
        self.optimizer.step()
        self.memory.clear()